# 03-3. 期待値・分散・共分散 — 動かして確かめる

📖 解説: [`../03_expectation_variance.md`](../03_expectation_variance.md)

## このノートで触るもの
1. 期待値 $\mathbb{E}[X]$
2. 分散 $\mathrm{Var}[X] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$
3. 共分散と相関係数
4. 共分散行列の可視化
5. 標準化

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (03_expectation_variance.md)](../03_expectation_variance.md)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, FloatSlider

%matplotlib inline
rng = np.random.default_rng(42)

## 1. 期待値 $\mathbb{E}[X]$

$$
\mathbb{E}[X] = \sum_i x_i \cdot P(X = x_i)
$$

対応するコード: `np.sum(xs * ps)` または `np.mean(samples)`

In [ ]:
# サイコロの期待値
xs = np.array([1, 2, 3, 4, 5, 6])
ps = np.full(6, 1/6)
ev = np.sum(xs * ps)
print(f'E[サイコロ] = {ev}')

# シミュレーションで確認
rolls = rng.integers(1, 7, size=1_000_000)
print(f'実測平均: {rolls.mean():.4f}')

## 2. 分散 $\mathrm{Var}[X]$

$$
\mathrm{Var}[X] = \mathbb{E}[(X - \mu)^2] = \mathbb{E}[X^2] - (\mathbb{E}[X])^2
$$

対応するコード: `np.var(x)` または `(x**2).mean() - x.mean()**2`

In [ ]:
data = rng.normal(0, 2, size=10_000)

print(f'E[X]        = {data.mean():.4f}')
print(f'Var[X]      = {data.var():.4f}')
print(f'σ           = {data.std():.4f}  (理論 σ=2)')
print()
print('別公式 E[X²] - (E[X])²:')
var_alt = (data**2).mean() - data.mean()**2
print(f'  = {var_alt:.4f}')

## 3. 共分散と相関係数

$$
\mathrm{Cov}[X, Y] = \mathbb{E}[(X - \mu_X)(Y - \mu_Y)]
$$

$$
\rho = \frac{\mathrm{Cov}[X, Y]}{\sigma_X \sigma_Y}
$$

対応するコード: `np.cov(x, y)`, `np.corrcoef(x, y)`

In [ ]:
def show_correlation(rho_target: float) -> None:
    """相関係数を変えて散布図."""
    n = 500
    x = rng.normal(0, 1, n)
    noise = rng.normal(0, 1, n)
    y = rho_target * x + np.sqrt(1 - rho_target**2) * noise
    
    actual_rho = np.corrcoef(x, y)[0, 1]
    
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(x, y, alpha=0.4, s=15)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(f'目標 $\\rho = {rho_target:.2f}$,  実測 $\\rho = {actual_rho:.4f}$')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    plt.show()

interact(show_correlation,
         rho_target=FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.7));

## 4. 共分散行列

$$
\Sigma_{ij} = \mathrm{Cov}[X_i, X_j]
$$

3 つの特徴量の例 (体重・身長・年齢):

In [ ]:
n = 1000
age = rng.uniform(20, 60, n)
height = 150 + 0.3 * (age - 40) + rng.normal(0, 10, n)   # 年齢で少し変動
weight = 0.6 * height - 50 + rng.normal(0, 8, n)          # 身長と強い相関

data = np.column_stack([age, height, weight])
feature_names = ['age', 'height', 'weight']

cov_matrix = np.cov(data, rowvar=False)
corr_matrix = np.corrcoef(data, rowvar=False)

print('共分散行列 Σ:')
print(np.round(cov_matrix, 2))
print('\n相関行列 ρ:')
print(np.round(corr_matrix, 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, m, title in zip(axes, [cov_matrix, corr_matrix], ['共分散行列', '相関行列']):
    im = ax.imshow(m, cmap='RdBu_r', vmin=-np.max(np.abs(m)), vmax=np.max(np.abs(m)))
    ax.set_xticks(range(3)); ax.set_xticklabels(feature_names)
    ax.set_yticks(range(3)); ax.set_yticklabels(feature_names)
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{m[i,j]:.2f}', ha='center', va='center')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 5. 標準化

$$
Z = \frac{X - \mu}{\sigma}
$$

対応するコード: `(x - x.mean()) / x.std()`

In [ ]:
# 異なるスケールの3特徴を標準化
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# 元データ (スケールがバラバラ)
axes[0].boxplot([age, height, weight], labels=feature_names)
axes[0].set_title('元データ (スケールがバラバラ)')
axes[0].grid(True, alpha=0.3)

# 標準化 (Z-score)
standardized = (data - data.mean(axis=0)) / data.std(axis=0)
axes[1].boxplot([standardized[:, i] for i in range(3)], labels=feature_names)
axes[1].set_title('標準化後 (平均 0, 標準偏差 1 に揃う)')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print('標準化後の各特徴量の平均と標準偏差:')
for i, name in enumerate(feature_names):
    z = standardized[:, i]
    print(f'  {name}: μ = {z.mean():.4f},  σ = {z.std():.4f}')

## まとめ
- 期待値は重心、分散は広がり
- 共分散は共変動、相関は規格化された共分散 ($\in [-1, 1]$)
- 共分散行列は ML の前処理 (PCA) や分布定義 (多変量正規) で必須
- 標準化は ML の前処理の定番

## 次へ
→ [`04_bayes.ipynb`](04_bayes.ipynb)  解説 → [`../04_bayes.md`](../04_bayes.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`02_distributions.ipynb`](02_distributions.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`04_bayes.ipynb`](04_bayes.ipynb) |